# 360° Rotated Object Detection Export Example

This notebook provides an end-to-end example on how to train and export a rotated object detection model that also predicts the **heading** of each object: which end is its front, over the full 360°. The model is exported to a format accepted by uniVision Module Image ONNX, so you can deploy it on the Wenglor's B60 Smart Camera and MVC Machine Vision Controller.

**180° or 360°?** The rotated object detection notebook (`rotated-object-detection-export-example.ipynb`) predicts boxes modulo 180°: a pencil pointing left and the same pencil pointing right give the same box. This notebook predicts the angle in [-180°, 180°), pointing from the box centre to the object's front. Use it when the direction matters, e.g. to pick a part tip first or to check that parts lie the right way round. It needs annotations that carry the heading; for everything else, use the 180° notebook.

DISCLAIMER: Focus of this notebook is to provide an example for the export and quantization. The training code is provided purely for illustration and not meant to provide a production-grade model. You will want to optimize the training pipeline and add more images. Depending on your situation, you can start from one of the following steps:
1. Train PyTorch model.
    * If you don't have a trained model, start here.
2. Export PyTorch model to ONNX.
    * If you already have a trained PyTorch model, start here.
3. Quantize ONNX model
    * If you have a trained ONNX model and it supports quantization, then start here. For this step, you will also need to provide a dataset that your model had been trained on. This dataset will be used for a calibration step during model quantization.
4. Export ONNX model.
    * If your ONNX model does not support quantization or you cannot provide a calibration dataset, begin here.

The models below have been tested and are known to run successfully:
* Rotated RTMDet-M-lite (default): mmrotate Rotated RTMDet-M with ReLU activations and a phase-shifting angle coder for the heading. The backbone, neck and most of the head are initialized from the RTMDet-M-lite COCO weights of the [TI edgeai model zoo](https://github.com/TexasInstruments/edgeai-tensorlab) (BSD-3-Clause / Apache-2.0); the class and angle layers are trained from scratch on your dataset.

We recommend a maximum input image size of 480×480 to stay within the memory and latency constraints of the B60 Smart Camera.

**What is quantization and why do we need it?**

Quantization is a technique used to make machine learning models run faster and more efficiently, especially on smaller devices with limited resources, like smartphones or smart cameras. By reducing the precision of the numbers the model uses, it can significantly decrease the amount of memory and computing power required, without losing too much accuracy. For example, our benchmark results show that a quantized ResNet50 model running on a B60 camera achieves an inference speed of 50 milliseconds, which is 8.4 times faster than the unquantized version (420 milliseconds).

## 1. Train PyTorch model

In [ ]:
# https://github.com/pytorch/pytorch/issues/140765
import getpass
import os

getpass.getuser = lambda: "torchuser"

os.environ["TORCHINDUCTOR_CACHE_DIR"] = "/tmp/torchinductor_cache"

In [ ]:
import copy
import json
import math
import os
import shutil
import time
import uuid
from functools import partial
from pathlib import Path

import cv2
import datumaro as dm
import numpy as np
import onnx
import onnxruntime as ort
import pandas as pd
import requests
import torch
import torch.nn as nn
import utils.rotated360 as r360
from mmdet.apis import init_detector
from mmengine.config import Config
from mmengine.runner import Runner
from mmrotate.utils import register_all_modules
from onnxruntime.quantization.quantize import (
    CalibrationMethod,
    QuantFormat,
    QuantType,
    quantize_static,
)
from onnxruntime.quantization.shape_inference import quant_pre_process
from sklearn.model_selection import train_test_split
from utils.enums import DatasetColorMode
from utils.export import export_univision_model_v4
from utils.image import (
    ensure_3ch_image,
    get_image_size,
    is_rgb_image,
    read_and_resize_input_example,
)
from utils.mixed_precision import compute_losses_in_float32
from utils.quantization import (
    TorchCalibrationDataReader,
    cap_channel_imbalance,
    find_final_convs,
    find_postprocess_nodes_to_exclude,
    find_score_path_nodes,
    keep_fp32_nodes_on_cpu,
    sort_nodes_topologically,
)
from utils.sparse_box_branch import (
    compute_box_branch_at_kept_candidates,
    quantize_features_with_relu_range,
)

register_all_modules()

In [ ]:
AI_INPUT_IMAGE_MAX_DIMENSION = 416
assert (
    AI_INPUT_IMAGE_MAX_DIMENSION % 32 == 0
), "AI_INPUT_IMAGE_SIZE must be divisible by 32"

# threshold to use to decide if a class probability enables the relative class
CLASS_THRESHOLD = 0.5

PAD_VALUE = (114, 114, 114)

DATA_ROOT = Path("../data")

# Your CVAT COCO export (annotations/ and images/)
ANNOTATION_FOLDER = DATA_ROOT / "coco-annotations/pencils"
EXAMPLE_IMAGE_FOLDER = DATA_ROOT / "images/multi-label/"
MODEL_FOLDER = DATA_ROOT / "model"

MMROTATE_DATA_FOLDER = DATA_ROOT / "mmrotate-360"
MMROTATE_OUTPUT_FOLDER = DATA_ROOT / "work_dirs" / "rotated_360_detection"

MMROTATE_DATA_FOLDER.mkdir(exist_ok=True)
MMROTATE_OUTPUT_FOLDER.mkdir(exist_ok=True, parents=True)
MODEL_FOLDER.mkdir(parents=True, exist_ok=True)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"The training will use {device} device")

In [ ]:
torch.load = partial(torch.load, weights_only=False)

### 1.1 Prepare dataset

This notebook expects a COCO-format dataset exported from CVAT with rotated bounding box annotations ("COCO 1.0" export).

**CVAT COCO 1.0 export format for rotated boxes:**
CVAT does NOT write rotated boxes as segmentation polygons. Instead it exports:
- `bbox`: `[x, y, w, h]` — the unrotated box: top-left corner, width and height
- `segmentation`: `[]` — empty
- `attributes.rotation`: rotation angle in degrees, clockwise, in [0, 360)

The rotation is the heading: the direction of the box's width axis, measured clockwise from the image's right-hand direction. Datumaro reads these as `dm.AnnotationType.bbox` objects with a `rotation` attribute. After the train/val split, each `bbox + rotation` is converted into a 4-point polygon whose first edge points along the heading (section 1.4).

Set `ANNOTATION_FOLDER` to the path of your COCO annotation folder. The folder must contain an `annotations/` subfolder with a COCO JSON file and an `images/` subfolder with the corresponding images. For the example dataset, the next cell copies its images into the folder.

In [ ]:
coco_dataset = dm.Dataset.import_from(
    path=ANNOTATION_FOLDER,
    format="coco_instances",
)

In [ ]:
# For custom dataset, please skip this
if ANNOTATION_FOLDER == DATA_ROOT / "coco-annotations/pencils":
    image_names = [Path(item.media.path).name for item in coco_dataset]
    image_source_paths = [
        next(EXAMPLE_IMAGE_FOLDER.rglob(image_name)) for image_name in image_names
    ]
    for image_source_path in image_source_paths:
        shutil.copy2(
            image_source_path, ANNOTATION_FOLDER / "images" / image_source_path.name
        )
else:
    print("Skipping image copy due to custom image dataset")

In [ ]:
index_to_class = {v: k for k, v in coco_dataset.categories()[1]._indices.items()}
classes = list(index_to_class.values())
classes

### 1.2 Visualize bounding boxes

Let's check the annotation to ensure that the dataset is read correctly. The arrow shows the annotated rotation, measured from the image's right-hand direction and increasing clockwise on screen (0° = right, 90° = down, 180° = left, 270° = up), whatever the shape of the box.

In [ ]:
coco_dataset_list = list(coco_dataset)
datumaro_bbox_folder = MMROTATE_OUTPUT_FOLDER / "datumaro_bbox"
datumaro_bbox_folder.mkdir(exist_ok=True)

for item in coco_dataset_list:
    image_path = Path(item.media.path)

    bbox_annotations = [
        ann for ann in item.annotations if ann.type == dm.AnnotationType.bbox
    ]
    boxes = np.array(
        [
            [
                ann.x + ann.w / 2,
                ann.y + ann.h / 2,
                ann.w,
                ann.h,
                math.radians(ann.attributes.get("rotation", 0.0)),
            ]
            for ann in bbox_annotations
        ]
    ).reshape(-1, 5)
    vis_image = r360.draw_directed_boxes(
        ensure_3ch_image(cv2.imread(str(image_path))),
        boxes,
        np.array([ann.label for ann in bbox_annotations]),
        class_names=classes,
        colour=(0, 160, 0),
    )
    cv2.imwrite(
        str(datumaro_bbox_folder / image_path.with_suffix(".jpg").name), vis_image
    )
print(f"Datumaro bounding boxes were visualized and saved to {datumaro_bbox_folder}")

### 1.3 Train-test-split

It is important to ensure that classes are equally distributed in train and validation. We will concatenate the class names so that we can do a stratified train-test-split. mmdetection expects the folder to be in the COCO format, we will transform our dataset to fit it.

```
dataset/
├── train/
│   ├── annotations/
│   │   └── instances_default.json
│   └── images/default/
│       ├── image1.jpg
│       ├── image2.jpg
│       └── ... (other image files)
└── valid/
    ├── annotations/
    │   └── instances_default.json
    └── images/default/
        ├── image1.jpg
        ├── image2.jpg
        └── ... (other image files)
```

In [ ]:
data = {}
for item in coco_dataset:
    image_id = item.id
    label_counts = {}
    for ann in item.annotations:
        if ann.type == dm.AnnotationType.bbox:
            label_name = index_to_class[ann.label]
            label_counts[label_name] = label_counts.get(label_name, 0) + 1

    data[image_id] = label_counts

df = pd.DataFrame.from_dict(data, orient="index").fillna(0).astype(int)
column_names = list(df.columns)

missing_class = [class_i for class_i in classes if class_i not in column_names]
if missing_class:
    print(
        f"Warning! {missing_class} are missing but are included in the annotation categories!"
    )

df = df.reset_index(names="image_id")
df["label_combo"] = df[column_names].apply(
    lambda row: "".join(str(bool(x)) for x in row), axis=1
)

df.head(3)

In [ ]:
df[["label_combo"]].value_counts()

In [ ]:
try:
    train_image_id, valid_image_id = train_test_split(
        df.image_id.values,
        test_size=0.2,
        random_state=0,
        stratify=df.label_combo,
    )
except ValueError:
    print("Warning: stratified split not possible, falling back to random split")
    train_image_id, valid_image_id = train_test_split(
        df.image_id.values,
        test_size=0.2,
        random_state=0,
    )
train_image_id.shape, valid_image_id.shape

In [ ]:
train = copy.deepcopy(coco_dataset)
valid = copy.deepcopy(coco_dataset)

for image_id in valid_image_id:
    train.remove(image_id)

for image_id in train_image_id:
    valid.remove(image_id)

In [ ]:
train.export("train", format="coco_instances", save_media=True)
valid.export("valid", format="coco_instances", save_media=True)

In [ ]:
for phase in ["train", "valid"]:
    folder_path = MMROTATE_DATA_FOLDER / phase
    if folder_path.exists():
        print(f"{folder_path} exists. Copy is skipped")
    else:
        Path(phase).rename(folder_path)

In [ ]:
# Fix duplicate annotation IDs produced by Datumaro export
for phase in ["train", "valid"]:
    ann_path = MMROTATE_DATA_FOLDER / phase / "annotations" / "instances_default.json"
    with open(ann_path) as f:
        coco_json = json.load(f)
    for new_id, ann in enumerate(coco_json["annotations"], start=1):
        ann["id"] = new_id
    with open(ann_path, "w") as f:
        json.dump(coco_json, f)
    print(f"{phase}: re-numbered {len(coco_json['annotations'])} annotation IDs")

### 1.4 Normalize rotation encoding to COCO segmentation polygons

mmrotate's data pipeline reads rotated boxes from the `segmentation` field as 4-point polygons. CVAT's own COCO export leaves `segmentation` empty and stores the angle in `attributes.rotation` instead. This cell is a CVAT-specific adapter that converts CVAT's encoding into polygons.

For the heading, the order of the corners matters: the polygon starts at the box's top-left corner before rotation, so its **first edge P1 → P2 points along the heading** (the box width, rotated by CVAT's rotation). The training pipeline reads the heading back from this edge (`ConvertMask2RBoxDirected`). A dataset that already provides segmentation polygons passes through unchanged; their first edge must then point along the heading as well.

In [ ]:
def cvat_bbox_rotation_to_polygon(x, y, w, h, angle_deg):
    """Convert a CVAT rotated bbox to a flat 4-point polygon.

    CVAT stores the rotated box as (x, y, w, h) — the unrotated box dimensions
    with top-left at (x, y) — plus a clockwise rotation angle in degrees applied
    around the box center (as seen on screen, y pointing down).

    Returns a flat list [x1, y1, x2, y2, x3, y3, x4, y4] suitable for the COCO
    segmentation field.
    """
    cx, cy = x + w / 2, y + h / 2
    hw, hh = w / 2, h / 2
    angle_rad = math.radians(angle_deg)
    cos_a = math.cos(angle_rad)
    sin_a = math.sin(angle_rad)

    corners = [(-hw, -hh), (hw, -hh), (hw, hh), (-hw, hh)]
    poly = []
    for dx, dy in corners:
        rx = cx + cos_a * dx - sin_a * dy
        ry = cy + sin_a * dx + cos_a * dy
        poly.extend([rx, ry])
    return poly


def convert_rotation_annotations(ann_path):
    """CVAT-specific adapter: normalize CVAT's `attributes.rotation` encoding
    into the COCO segmentation-polygon convention this pipeline expects
    downstream. Annotations that already have a segmentation polygon (e.g. a
    dataset converted from DOTA) pass through unchanged.

    Returns (converted, unhandled). `unhandled` counts annotations with
    neither a CVAT rotation attribute nor an existing polygon -- i.e. a
    rotation encoding this notebook doesn't recognize yet.
    """
    with open(ann_path) as f:
        coco = json.load(f)

    converted = 0
    unhandled = 0
    for ann in coco["annotations"]:
        if ann.get("segmentation"):
            continue
        rotation = ann.get("attributes", {}).get("rotation")
        if rotation is None:
            unhandled += 1
            continue
        x, y, w, h = ann["bbox"]
        poly = cvat_bbox_rotation_to_polygon(x, y, w, h, rotation)
        ann["segmentation"] = [poly]
        xs, ys = poly[0::2], poly[1::2]
        ann["bbox"] = [min(xs), min(ys), max(xs) - min(xs), max(ys) - min(ys)]
        converted += 1

    with open(ann_path, "w") as f:
        json.dump(coco, f)
    return converted, unhandled


for phase in ["train", "valid"]:
    ann_path = MMROTATE_DATA_FOLDER / phase / "annotations" / "instances_default.json"
    converted, unhandled = convert_rotation_annotations(ann_path)
    print(
        f"{phase}: converted {converted} annotations to 4-point segmentation polygons"
    )
    if unhandled:
        raise ValueError(
            f"{phase}: {unhandled} annotation(s) have neither a CVAT "
            "`attributes.rotation` nor an existing segmentation polygon -- "
            "this dataset's rotation encoding isn't recognized. Add an "
            "adapter for it here, alongside cvat_bbox_rotation_to_polygon."
        )

Check that the labels carry a heading: a dataset annotated for the 180° notebook typically has all its angles within ±90°.

In [ ]:
for phase in ["train", "valid"]:
    headings, aspect_ratios = r360.coco_headings(
        MMROTATE_DATA_FOLDER / phase / "annotations" / "instances_default.json"
    )
    histogram = np.histogram(headings, bins=8, range=(-180, 180))[0]
    print(
        f"{phase}: {len(headings)} boxes, headings per 45° from -180°: {histogram.tolist()}"
    )
    if (np.abs(headings) <= 90).all():
        print(
            "Warning: all headings lie within ±90°. Do the annotations carry the heading?"
        )

### 1.5 Find image size with minimal padding

The accuracy of bounding box tends to improve when the image is not distorted, we will therefore resize the image while preserving the aspect ratio, the AI input image size will be achieved by padding. The following section will find the maximum image size of this dataset and calculate the image size with minimal padding based on the specified `AI_INPUT_IMAGE_MAX_DIMENSION` for optimal inference and training time. 

In [ ]:
image_sizes = []
for item in coco_dataset:
    image_sizes.append(get_image_size(item.media.path))
print(f"unique image sizes: {set(image_sizes)}")
image_sizes = np.array(image_sizes)

max_image_size = (np.max(image_sizes[:, 0]), np.max(image_sizes[:, 1]))

In [ ]:
def get_image_size_with_minimal_padding(
    original_size: tuple, max_dimension: int
) -> tuple:
    """
    Get image size with minimal padding that is divisible by 32 and has max_dimension as one dimension.

    Args:
        original_size: Original (width, height) tuple
        max_dimension: Maximum allowed dimension (e.g., 640, 512, 768)

    Returns:
        New (width, height) tuple that is divisible by 32 and minimizes padding
    """
    if max_dimension % 32 != 0:
        raise ValueError("Max dimension must be divisible by 32.")
    orig_height, orig_width = original_size
    aspect_ratio = orig_width / orig_height

    if orig_height > orig_width:
        new_height = max_dimension
        new_width = int(np.ceil(new_height * aspect_ratio / 32) * 32)
    else:
        new_width = max_dimension
        new_height = int(np.ceil(new_width / aspect_ratio / 32) * 32)

    return (new_height, new_width)

In [ ]:
AI_INPUT_IMAGE_SIZE = get_image_size_with_minimal_padding(
    max_image_size, AI_INPUT_IMAGE_MAX_DIMENSION
)
print(f"AI_INPUT_IMAGE_SIZE: {AI_INPUT_IMAGE_SIZE}")

### 1.6 Train model

mmrotate trains models with a configuration-based system, for more details click [here](https://mmrotate.readthedocs.io/en/latest/). This notebook trains **Rotated RTMDet-M** with ReLU activations, like the 180° notebook, with specific changes to handle 360 rotation.

In [ ]:
# Rotated RTMDet-M with ReLU activations, initialized from the RTMDet-M-lite COCO weights
# of the TI edgeai model zoo (BSD-3-Clause / Apache-2.0)
model_cfg = {
    "config": "rotated_rtmdet/rotated_rtmdet_m-3x-dota.py",
    "checkpoint": "rtmdet_m_coco_lite_640x640_20250404_checkpoint.pth",
    "url": "https://software-dl.ti.com/jacinto7/esd/modelzoo/11_02_00/models/vision/detection/coco/edgeai-mmdet/rtmdet_m_coco_lite_640x640_20250404_checkpoint.pth",
    "input_color_space": "BGR",
    "unit_scaling": False,
    "mean": (103.53, 116.28, 123.675),
    "std": (57.375, 57.12, 58.395),
}
default_model_config_path = Path(f"./mmrotate/configs/{model_cfg['config']}")
pretrained_checkpoint_path = MODEL_FOLDER / model_cfg["checkpoint"]


def use_relu_activations(model):
    """Uses ReLU in the channel attention layers of RTMDet (the config sets all other activations)."""
    for module in model.modules():
        if type(module).__name__ == "ChannelAttention":
            module.act = nn.ReLU()
    return model

In [ ]:
if not pretrained_checkpoint_path.exists():
    print("Downloading checkpoint")
    with requests.get(model_cfg["url"], stream=True) as response:
        response.raise_for_status()
        with open(pretrained_checkpoint_path, "wb") as f:
            for chunk in response.iter_content(chunk_size=8192):
                if chunk:
                    f.write(chunk)
    print(f"Checkpoint downloaded successfully to {pretrained_checkpoint_path}")
else:
    print("Checkpoint already exists, skipping download")

In [ ]:
num_train_images = len(train_image_id)

BATCH_SIZE = min(16, num_train_images // 4)
steps_per_epoch = num_train_images // BATCH_SIZE

# Target ~2500 total training steps for finetuning
target_total_steps = 2500
EPOCHS = max(60, target_total_steps // steps_per_epoch)

# AdamW LR with linear scaling rule
base_lr = 0.00025
effective_lr = base_lr * (BATCH_SIZE / 16)
weight_decay = 0.01

backbone_lr_ratio = 1.0
neck_lr_ratio = 1.0
head_lr_ratio = 1.0

# Optional scale augmentation, e.g. (0.75, 1.25): random zoom of the training images. Off by
# default: with a fixed camera distance the objects always have the same size, and the zoom
# cost accuracy in our tests. Try it when object sizes vary in your images.
SCALE_JITTER = None

print(f"Dataset size:      {num_train_images} images")
print(f"Batch size:        {BATCH_SIZE}")
print(f"Steps per epoch:   {steps_per_epoch}")
print(f"Epochs:            {EPOCHS}")
print(f"Total steps:       {EPOCHS * steps_per_epoch}")
print(f"Effective LR:      {effective_lr}")
print(f"Weight decay:      {weight_decay}")
print(f"Backbone LR ratio: {backbone_lr_ratio}")
print(f"Neck LR ratio:     {neck_lr_ratio}")
print(f"Head LR ratio:     {head_lr_ratio}")

In [ ]:
# mmdet transforms take (width, height); AI_INPUT_IMAGE_SIZE is (height, width)
img_scale = AI_INPUT_IMAGE_SIZE[::-1]
annotation_json_relative_path = Path("annotations/instances_default.json")
images_relative_path = Path("images/default/")
data_root = str(MMROTATE_DATA_FOLDER)

In [ ]:
cfg = Config.fromfile(str(default_model_config_path))
cfg.backend_args = None

# --- Pipelines ---
train_pipeline = [
    dict(type="mmdet.LoadImageFromFile", backend_args=None),
    dict(type="mmdet.LoadAnnotations", with_bbox=True, with_mask=True, poly2mask=False),
    # boxes with their heading, read from the polygon's first edge
    dict(type="ConvertMask2RBoxDirected"),
    # fit inside the model input, also for images with another aspect ratio; pads bottom/right
    dict(
        type="mmdet.FixShapeResize",
        width=img_scale[0],
        height=img_scale[1],
        keep_ratio=True,
        pad_val=dict(img=PAD_VALUE),
    ),
    # optional scale jitter of the fitted frame, cropped back to the input size
    *(
        [
            dict(
                type="mmdet.RandomResize",
                resize_type="mmdet.Resize",
                scale=img_scale,
                ratio_range=SCALE_JITTER,
                keep_ratio=True,
            ),
            dict(type="mmdet.RandomCrop", crop_size=img_scale),
        ]
        if SCALE_JITTER
        else []
    ),
    # only the diagonal flip (a 180° rotation): a mirrored object can look different
    dict(type="mmdet.RandomFlip", prob=0.5, direction=["diagonal"]),
    dict(type="mmdet.Pad", size=img_scale, pad_val=dict(img=PAD_VALUE)),
    # rotation over the full circle ([-180°, 180°))
    dict(type="RandomRotate", prob=0.9, angle_range=180),
    # HSV jitter: hue/saturation/value shifts — image-only, safe with rotated boxes.
    dict(type="mmdet.YOLOXHSVRandomAug"),
    dict(type="mmdet.PackDetInputs"),
]

val_pipeline = [
    dict(type="mmdet.LoadImageFromFile", backend_args=None),
    dict(
        type="mmdet.FixShapeResize",
        width=img_scale[0],
        height=img_scale[1],
        keep_ratio=True,
        pad_val=dict(img=PAD_VALUE),
    ),
    dict(type="mmdet.LoadAnnotations", with_bbox=True, with_mask=True, poly2mask=False),
    dict(type="ConvertMask2RBoxDirected"),
    dict(
        type="mmdet.PackDetInputs",
        meta_keys=(
            "img_id",
            "img_path",
            "ori_shape",
            "img_shape",
            "scale_factor",
            "instances",
        ),
    ),
]

inference_pipeline_cfg = [
    dict(type="mmdet.LoadImageFromFile", backend_args=None),
    dict(
        type="mmdet.FixShapeResize",
        width=img_scale[0],
        height=img_scale[1],
        keep_ratio=True,
        pad_val=dict(img=PAD_VALUE),
    ),
    dict(type="mmdet.PackDetInputs"),
]

# --- Dataloaders ---
cfg.train_dataloader = dict(
    batch_size=BATCH_SIZE,
    num_workers=4,
    persistent_workers=True,
    sampler=dict(type="DefaultSampler", shuffle=True),
    dataset=dict(
        type="mmdet.CocoDataset",
        data_root=data_root,
        ann_file=str(Path("train") / annotation_json_relative_path),
        data_prefix=dict(img=str(Path("train") / images_relative_path)),
        metainfo=dict(classes=classes),
        filter_cfg=dict(filter_empty_gt=True, min_size=32),
        pipeline=train_pipeline,
        backend_args=None,
    ),
)

cfg.val_dataloader = dict(
    batch_size=BATCH_SIZE,
    num_workers=4,
    persistent_workers=True,
    drop_last=False,
    sampler=dict(type="DefaultSampler", shuffle=False),
    dataset=dict(
        type="mmdet.CocoDataset",
        data_root=data_root,
        ann_file=str(Path("valid") / annotation_json_relative_path),
        data_prefix=dict(img=str(Path("valid") / images_relative_path)),
        metainfo=dict(classes=classes),
        test_mode=True,
        pipeline=val_pipeline,
        backend_args=None,
    ),
)
cfg.test_dataloader = cfg.val_dataloader

# AP50 with rotated IoU, plus dAP50: a match also needs a heading error below 30°
cfg.val_evaluator = dict(type="DirectedRotatedMetric", class_names=classes)
cfg.test_evaluator = cfg.val_evaluator

# --- Model overrides ---
cfg.model.bbox_head.num_classes = len(classes)
# heading over the full circle: angle range, phase-shifting angle coder and angle loss
cfg.model.bbox_head.angle_version = "r360"
cfg.model.bbox_head.bbox_coder.angle_version = "r360"
cfg.model.bbox_head.angle_coder = dict(type="PSC360Coder", freqs=(1, 2), detach=True)
cfg.model.bbox_head.loss_angle = dict(type="mmdet.L1Loss", loss_weight=1.0)

for part in ("backbone", "neck", "bbox_head"):
    cfg.model[part].act_cfg = dict(type="ReLU", inplace=True)
cfg.model.backbone.init_cfg = None

# --- Training schedule ---
cfg.train_cfg.max_epochs = EPOCHS
# validate in the second half only, about 8 times: the best checkpoint comes from there
cfg.train_cfg.val_interval = max(1, EPOCHS // 15)
cfg.train_cfg.val_begin = EPOCHS // 2

# --- Optimizer: AdamW ---
# Mixed precision (float16 convolutions) trains faster on the GPU;
# the losses stay in float32 (compute_losses_in_float32 below)
use_mixed_precision = device.type == "cuda"
cfg.optim_wrapper = dict(
    type="AmpOptimWrapper" if use_mixed_precision else "OptimWrapper",
    **(dict(loss_scale="dynamic") if use_mixed_precision else {}),
    optimizer=dict(type="AdamW", lr=effective_lr, weight_decay=weight_decay),
    paramwise_cfg=dict(
        custom_keys={
            "backbone": dict(lr_mult=backbone_lr_ratio, decay_mult=1.0),
            "neck": dict(lr_mult=neck_lr_ratio, decay_mult=1.0),
            "bbox_head": dict(lr_mult=head_lr_ratio, decay_mult=1.0),
        },
        bypass_duplicate=True,
    ),
    clip_grad=dict(max_norm=35, norm_type=2),
)

# --- LR scheduler: linear warmup, constant for the first half, cosine annealing in the second ---
cfg.param_scheduler = [
    dict(
        type="LinearLR",
        start_factor=1.0 / 3,
        by_epoch=False,
        begin=0,
        end=500,
    ),
    dict(
        type="CosineAnnealingLR",
        eta_min=effective_lr * 0.01,
        begin=EPOCHS // 2,
        T_max=EPOCHS - EPOCHS // 2,
        end=EPOCHS,
        by_epoch=True,
        convert_to_iter_based=True,
    ),
]

# --- Hooks ---
cfg.default_hooks.logger.interval = 50
cfg.default_hooks.checkpoint = dict(
    type="CheckpointHook",
    save_best="dir/dAP50",
    rule="greater",
    interval=EPOCHS + 1,
    max_keep_ckpts=1,
    save_last=False,
)

cfg.custom_hooks = [
    dict(
        type="EMAHook",
        ema_type="mmdet.ExpMomentumEMA",
        momentum=0.0002,
        update_buffers=True,
        priority=49,
    ),
]

cfg.test_pipeline = inference_pipeline_cfg
cfg.work_dir = str(MMROTATE_OUTPUT_FOLDER)
cfg.load_from = str(pretrained_checkpoint_path)
cfg.resume = False
cfg.env_cfg.cudnn_benchmark = True

In [ ]:
runner = Runner.from_cfg(cfg)
use_relu_activations(runner.model)
if use_mixed_precision:
    compute_losses_in_float32(runner.model)
runner.train()

### 1.7 Evaluate results

The predictions of the trained model on the validation images, drawn with their heading.

In [ ]:
config_file_path = MMROTATE_OUTPUT_FOLDER / default_model_config_path.name
checkpoint_file_path = sorted(MMROTATE_OUTPUT_FOLDER.glob("best_dir_dAP50_epoch_*"))[-1]
model = init_detector(str(config_file_path), str(checkpoint_file_path), device="cpu")
use_relu_activations(model)
model.eval()
print(f"Model loaded from {checkpoint_file_path}")

In [ ]:
valid_images_dir = MMROTATE_DATA_FOLDER / "valid" / "images" / "default"

valid_image_paths = [
    p
    for p in valid_images_dir.rglob("*")
    if p.is_file()
    and p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".avif"}
]

In [ ]:
raw_predictions_folder = MMROTATE_OUTPUT_FOLDER / "raw_prediction"
raw_predictions_folder.mkdir(exist_ok=True)

# The export wrapper of section 2, run in PyTorch: the same post-processing as the ONNX model
torch_predict = r360.torch_predict_fn(
    r360.RotatedDetectionExport(copy.deepcopy(model)).eval(),
    AI_INPUT_IMAGE_SIZE,
    model_cfg,
    device="cpu",
)
for image_path in valid_image_paths:
    image = ensure_3ch_image(cv2.imread(str(image_path)))
    boxes, labels, scores = torch_predict(image)
    vis_image = r360.draw_directed_boxes(
        image, boxes, labels, scores, class_names=classes, score_thr=CLASS_THRESHOLD
    )
    cv2.imwrite(
        str(raw_predictions_folder / image_path.with_suffix(".jpg").name), vis_image
    )
print(f"Raw predictions were visualized and saved to {raw_predictions_folder}")

## 2. Export PyTorch Model to ONNX

If you already have a trained PyTorch model, start here.

**Note:** Quantizing for **NPU** acceleration may fail if your model contains unsupported layers/operators.
Only the models listed at the start of this notebook are guaranteed to work in the full NPU workflow.

Alternatively, you can quantize and run on **CPU** instead. This still gives significant speed-ups with much broader layer support. See `inference_device` in [metadata.md](../metadata.md), this can be specifed at the final step of this notebook.

The module image ONNX expects the ONNX model output to be as follows:
* boxes: The shape of the corresponding output must be `(detections_count, 5)` for oriented boxes, the type is float32. The 5 values are `(cx, cy, w, h, angle_rad)` (see `boxes_format="center_size_angle"` in §4). Here the angle is the **heading**: in [-π, π), clockwise in image coordinates, 0 when the heading points to the right of the image; `w` is the box edge along the heading.
* labels: The shape of the corresponding output must be `(detections_count,)`, the type is int64. Each value must be in the range `[0..classes-1]`.
* scores: The shape of the corresponding output must be `(detections_count,)`, the type is float32. Each value must be in the range `[0..1]`.

The export wrapper (`RotatedDetectionExport` in `utils/rotated360.py`) is the one of the 180° notebook: it thresholds the class scores, decodes the boxes of the kept candidates (the angle coder turns the phase-shifting code into the heading) and runs NMS.

In [ ]:
FP32_ONNX_MODEL_PATH = MODEL_FOLDER / "rotated_360_fp32_model.onnx"

In [ ]:
wrapper = r360.RotatedDetectionExport(
    base_model=model,
    conf_thre=0.05,
    nms_iou_thr=0.5,
    max_per_img=100,
    max_candidates=200,
)

wrapper.eval()

dummy_input = torch.randn(1, 3, AI_INPUT_IMAGE_SIZE[0], AI_INPUT_IMAGE_SIZE[1])
torch.onnx.export(
    wrapper,
    dummy_input,
    FP32_ONNX_MODEL_PATH,
    input_names=["input"],
    output_names=["boxes", "labels", "scores"],
    opset_version=20,
    do_constant_folding=True,
    dynamo=True,
    external_data=False,
    optimize=True,
)
print(f"ONNX FP32 model exported to {FP32_ONNX_MODEL_PATH}")

### 2.1 Evaluate the ONNX FP32 model

We will use the exported model to predict the validation dataset to verify the correctness of our wrapper. The metrics:

* **AP50**: average precision with a rotated IoU of at least 0.5. The rotated IoU ignores the heading.
* **dAP50**: as AP50, but a match also needs a heading error below 30°.
* **dir_acc**: the share of matches with a heading error below 90°, i.e. pointing the right way.
* **mae**: the mean heading error of the matches, in degrees.

The preprocessing (`r360.preprocess`: resize keeping the aspect ratio, pad bottom/right with `PAD_VALUE`, normalise) matches the training and uniVision, as in the 180° notebook.

In [ ]:
valid_ground_truth = r360.load_coco_gt(MMROTATE_DATA_FOLDER, "valid")
METRICS = ["AP50", "dAP50", "dir_acc", "mae"]


def evaluate_onnx_model(model_path):
    # Basic graph optimisation only: ONNX Runtime then skips its own int8 kernels, which can
    # saturate on x86 CPUs without VNNI and show a loss that is not in the quantized model.
    predict = r360.onnx_predict_fn(
        model_path, AI_INPUT_IMAGE_SIZE, model_cfg, optimization="basic"
    )
    return r360.evaluate_predict_fn(predict, valid_ground_truth, classes)


def save_onnx_predictions(model_path, image_paths, output_folder, label):
    output_folder.mkdir(exist_ok=True)
    predict = r360.onnx_predict_fn(model_path, AI_INPUT_IMAGE_SIZE, model_cfg)
    for image_path in image_paths:
        image = ensure_3ch_image(cv2.imread(str(image_path)))
        boxes, labels, scores = predict(image)
        vis_image = r360.draw_directed_boxes(
            image, boxes, labels, scores, class_names=classes, score_thr=CLASS_THRESHOLD
        )
        cv2.imwrite(str(output_folder / image_path.with_suffix(".jpg").name), vis_image)
    print(f"Predictions of the {label} ONNX model were saved to {output_folder}.")


fp32_metrics = evaluate_onnx_model(FP32_ONNX_MODEL_PATH)
print({name: round(fp32_metrics[name], 4) for name in METRICS})
save_onnx_predictions(
    FP32_ONNX_MODEL_PATH,
    valid_image_paths,
    MMROTATE_OUTPUT_FOLDER / "fp32_onnx_predictions",
    label="FP32",
)

## 3. Quantize ONNX model (optional)

This step is required to utilize the NPU at B60, which dramatically speeds up model inference, the drawback is a potential degradation of model quality. This notebook uses MinMax quantization with per-tensor weights, together with extra steps to preserve model quality, as the 180° notebook, plus two changes that make the calibration robust:

* **Moving-average MinMax calibration** (`CalibMovingAverage`): the ranges are averaged over the calibration images instead of taken from the most extreme one, so a few unusual images can't stretch them.
* **The class logits stay float32**, from the class prediction layers to the score (`find_score_path_nodes`); with averaged ranges they would otherwise be clipped, capping the scores.

The calibration order is seeded, so the result is reproducible. Layers that are not quantized run on the CPU, the rest stays hardware-accelerated (`keep_fp32_nodes_on_cpu`).

We recommend excluding the NMS from quantization for 2 purposes:
* Preserve postprocessing quality
* Ensure NPU compability

In [ ]:
INT8_ONNX_MODEL_PATH = MODEL_FOLDER / "rotated_360_int8_model.onnx"
NUM_WORKERS = 4
CALIBRATION_SAMPLE_SIZE = 200  # Please adjust this according to your usecase.

In [ ]:
class CalibrationDataset(torch.utils.data.Dataset):
    def __init__(self, image_paths):
        self.image_paths = image_paths

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image = ensure_3ch_image(cv2.imread(str(self.image_paths[idx])))
        return r360.preprocess(image, AI_INPUT_IMAGE_SIZE, model_cfg)[0], 0


train_images_dir = MMROTATE_DATA_FOLDER / "train" / "images" / "default"
train_image_paths = sorted(
    p
    for p in train_images_dir.rglob("*")
    if p.is_file()
    and p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".avif"}
)
calibration_dataset = CalibrationDataset(train_image_paths)

In [ ]:
float32_capped_model_path = Path(
    str(FP32_ONNX_MODEL_PATH).replace(".onnx", "_capped.onnx")
)
float32_preprocessed_model_path = Path(
    str(FP32_ONNX_MODEL_PATH).replace(".onnx", "_preprocessed.onnx")
)

print("Quantizing ONNX model")

# Weights are quantized per tensor: cap the imbalance between the output channels of each
# layer, so that small channels keep their precision.
onnx_model = onnx.load(FP32_ONNX_MODEL_PATH)
tensors_changed, channels_rescaled = cap_channel_imbalance(
    onnx_model, max_ratio=3.0, compensate=True
)
print(f"Rescaled {channels_rescaled} channels in {tensors_changed} layers")

# The last box and angle layers run only at the candidates above the confidence threshold,
# so that they can stay in float32.
sparse_box_branch = compute_box_branch_at_kept_candidates(onnx_model)
onnx.save(onnx_model, float32_capped_model_path)

quant_pre_process(
    input_model_path=float32_capped_model_path,
    output_model_path=float32_preprocessed_model_path,
    skip_symbolic_shape=True,
)

# Box decoding and NMS stay float32: backbone, neck and head form one quantized subgraph (ideal for NPU).
postprocess_nodes = find_postprocess_nodes_to_exclude(float32_preprocessed_model_path)
print(f"Excluding {len(postprocess_nodes)} post-processing nodes from quantization")

# The final class prediction layers and the class logits up to the score stay in float32
prediction_nodes = [
    n
    for n in find_final_convs(float32_preprocessed_model_path)
    if n not in sparse_box_branch.feature_convs
]
score_nodes = find_score_path_nodes(float32_preprocessed_model_path, prediction_nodes)
print(
    f"Excluding {len(prediction_nodes)} prediction layers and {len(score_nodes)} score nodes"
)

torch.manual_seed(0)  # reproducible calibration order
calibration_data_reader = TorchCalibrationDataReader(
    float32_preprocessed_model_path,
    samples=CALIBRATION_SAMPLE_SIZE,
    dataset=calibration_dataset,
    batch_size=1,
    shuffle=True,
    num_workers=NUM_WORKERS,
    persistent_workers=NUM_WORKERS > 0,
)

quantize_static(
    model_input=float32_preprocessed_model_path,
    model_output=INT8_ONNX_MODEL_PATH,
    calibration_data_reader=calibration_data_reader,
    quant_format=QuantFormat.QDQ,
    activation_type=QuantType.QUInt8,
    weight_type=QuantType.QInt8,
    per_channel=False,
    calibrate_method=CalibrationMethod.MinMax,
    nodes_to_exclude=postprocess_nodes
    + prediction_nodes
    + score_nodes
    + sparse_box_branch.new_nodes,
    extra_options={"CalibMovingAverage": True},
)
# The layers excluded from quantization run on the CPU, the rest stays hardware-accelerated
onnx_model = onnx.load(INT8_ONNX_MODEL_PATH)
sort_nodes_topologically(onnx_model)
onnx_model = keep_fp32_nodes_on_cpu(onnx_model)
# the gathered features are quantized with the range of the ReLU that follows them
quantize_features_with_relu_range(onnx_model, sparse_box_branch.feature_convs)
onnx.save(onnx_model, INT8_ONNX_MODEL_PATH)
print(f"Model quantized and saved to: {INT8_ONNX_MODEL_PATH}")

### 3.1 Compare FP32 and INT8 models

Quantization might lead to a degradation in model quality, it is important to verify the quality of the quantized model by comparing it against the FP32 model. Note that quantization is optional, if the model quality is not acceptable you can always run a FP32 model as long as it fits your inference time requirements.

In [ ]:
save_onnx_predictions(
    INT8_ONNX_MODEL_PATH,
    valid_image_paths,
    MMROTATE_OUTPUT_FOLDER / "int8_onnx_predictions",
    label="INT8",
)

int8_metrics = evaluate_onnx_model(INT8_ONNX_MODEL_PATH)

print(f"\n{'='*45}")
print("  FP32 vs INT8 Comparison")
print(f"{'='*45}")
print(f"  {'Metric':<12} {'FP32':>8} {'INT8':>8} {'Δ':>8}")
print(f"  {'-'*12} {'-'*8} {'-'*8} {'-'*8}")
for name in METRICS:
    fp32_val, int8_val = fp32_metrics[name], int8_metrics[name]
    print(
        f"  {name:<12} {fp32_val:>8.4f} {int8_val:>8.4f} {int8_val - fp32_val:>+8.4f}"
    )

## 4. Export ONNX model

### 4.1 YAML format

During model export, a metadata .yaml file is generated, it contains all the necessary information about the model such as the input size, class names, etc. It is essential to use the right parameters to ensure that correctness of the preprocessing. Please review the example metadata file and its description below. For more details, refer to [metadata.md](../metadata.md)

The metadata is the same as for the 180° notebook; only the angle in the boxes output covers [-π, π) and gives the heading.

#### 4.1.1 `.yaml` metadata file example

#### 4.1.2 Input preprocessing order

1. (optional) Color space/channel transformation

    As a result of this transformation the image data has shape
    
        a. (batch_size, original_height, original_width, channels) for channel_order=NHWC
      
        b. (batch_size, channels, original_height, original_width) for channel_order=NCHW

   where channels is 1 or 3 and the image channel data matches the order defined by color_space (RGB/BGR/GRAYSCALE).

3. Image resize

4. (optional) Unit scaling.

    `x = x / 255`

   
5. (optional) Standardization

   `x = (x - standardization_mean) / standardization_std`

### 4.2 Export INT8 model

In [ ]:
UNIVISION_INT8_MODEL_PATH = MODEL_FOLDER / "rotated_360_int8_model.u3o"

In [ ]:
if any(is_rgb_image(path) for path in valid_image_paths):
    dataset_color_mode = DatasetColorMode.COLOR
else:
    dataset_color_mode = DatasetColorMode.MONOCHROME

dataset_color_mode

In [ ]:
input_example = read_and_resize_input_example(
    valid_image_paths[0], AI_INPUT_IMAGE_SIZE, dataset_color_mode
)

In [ ]:
export_univision_model_v4(
    univision_model_path=UNIVISION_INT8_MODEL_PATH,
    onnx_model_path=INT8_ONNX_MODEL_PATH,
    classes=classes,
    input_example=input_example,
    model_name=None,
    model_uuid=str(uuid.uuid4()),
    inference_device="AUTO",
    quantization="INT8",
    resize_mode="FIT_WITH_PADDING",
    resize_padding_value=PAD_VALUE,
    resize_image_alignment_horizontal="LEFT",
    resize_image_alignment_vertical="TOP",
    unit_scaling=model_cfg["unit_scaling"],
    standardization_std=model_cfg["std"],
    standardization_mean=model_cfg["mean"],
    channel_order="NCHW",
    dataset_color_mode=dataset_color_mode,
    input_color_space=model_cfg["input_color_space"],
    output_type="ROTATED_OBJECT_DETECTION",
    class_thresholds=[CLASS_THRESHOLD] * len(classes),
    boxes_output_index=0,
    labels_output_index=1,
    scores_output_index=2,
    boxes_format="center_size_angle",
    boxes_coordinates="absolute",
    angle_unit="radians",
    max_detections=20,
)

### 4.3 Export FP32 model

In [ ]:
UNIVISION_FP32_MODEL_PATH = MODEL_FOLDER / "rotated_360_fp32_model.u3o"

In [ ]:
if any(is_rgb_image(path) for path in valid_image_paths):
    dataset_color_mode = DatasetColorMode.COLOR
else:
    dataset_color_mode = DatasetColorMode.MONOCHROME

dataset_color_mode

In [ ]:
input_example = read_and_resize_input_example(
    valid_image_paths[0], AI_INPUT_IMAGE_SIZE, dataset_color_mode
)

In [ ]:
export_univision_model_v4(
    univision_model_path=UNIVISION_FP32_MODEL_PATH,
    onnx_model_path=FP32_ONNX_MODEL_PATH,
    classes=classes,
    input_example=input_example,
    model_name=None,
    model_uuid=str(uuid.uuid4()),
    inference_device="AUTO",
    quantization=None,
    resize_mode="FIT_WITH_PADDING",
    resize_padding_value=PAD_VALUE,
    resize_image_alignment_horizontal="LEFT",
    resize_image_alignment_vertical="TOP",
    unit_scaling=model_cfg["unit_scaling"],
    standardization_std=model_cfg["std"],
    standardization_mean=model_cfg["mean"],
    channel_order="NCHW",
    dataset_color_mode=dataset_color_mode,
    input_color_space=model_cfg["input_color_space"],
    output_type="ROTATED_OBJECT_DETECTION",
    class_thresholds=[CLASS_THRESHOLD] * len(classes),
    boxes_output_index=0,
    labels_output_index=1,
    scores_output_index=2,
    boxes_format="center_size_angle",
    boxes_coordinates="absolute",
    angle_unit="radians",
    max_detections=20,
)